# U.S. Hospital Quality Analysis

This project analyzes hospital quality across the United States using publicly available data from the Centers for Medicare & Medicaid Services (CMS).

## Data Source

Center for Medicare & Medicaid Services (CMS)
Provider Data Catalog - Hospital General Information
Dataset ID: xubh-q36u
Source Page: https://data.cms.gov/provider-data/dataset/xubh-q36u
Access Method: CMS Provider Data Catalog API 

In [33]:
import pandas as pd
import matplotlib.pyplot as plt
import requests
import sqlite3
print("pandas:", pd.__version__)
print("Environment ready")

pandas: 3.0.5
Environment ready


In [34]:
from datetime import date
import pandas as pd
import requests

DATASET_ID = "xubh-q36u"
API_URL = (
    "https://data.cms.gov/provider-data/api/1/datastore/query/"
    f"{DATASET_ID}/0"
)
BATCH_SIZE = 1500

records = []
offset = 0

while True:
    response = requests.get(
        API_URL,
        params={"offset": offset, "limit": BATCH_SIZE},
        timeout=60,
    )
    response.raise_for_status()
    payload = response.json()
    batch = payload.get("results", [])

    if not batch:
        break

    records.extend(batch)
    print(f"Retrieved {len(records):,} rows")

    if len(batch) < BATCH_SIZE: break

    offset += BATCH_SIZE

    df_raw = pd.DataFrame(records)
    access_date = date.today().isoformat()

    print("Access date", access_date)
    print("Shape", df_raw.shape)
    df_raw.head()



Retrieved 1,500 rows
Access date 2026-07-26
Shape (1500, 38)
Retrieved 3,000 rows
Access date 2026-07-26
Shape (3000, 38)
Retrieved 4,500 rows
Access date 2026-07-26
Shape (4500, 38)
Retrieved 5,432 rows


In [35]:
print(df_raw.shape)
print(df_raw.columns.tolist())
df_raw.info()


(4500, 38)
['facility_id', 'facility_name', 'address', 'citytown', 'state', 'zip_code', 'countyparish', 'telephone_number', 'hospital_type', 'hospital_ownership', 'emergency_services', 'meets_criteria_for_birthing_friendly_designation', 'hospital_overall_rating', 'hospital_overall_rating_footnote', 'mort_group_measure_count', 'count_of_facility_mort_measures', 'count_of_mort_measures_better', 'count_of_mort_measures_no_different', 'count_of_mort_measures_worse', 'mort_group_footnote', 'safety_group_measure_count', 'count_of_facility_safety_measures', 'count_of_safety_measures_better', 'count_of_safety_measures_no_different', 'count_of_safety_measures_worse', 'safety_group_footnote', 'readm_group_measure_count', 'count_of_facility_readm_measures', 'count_of_readm_measures_better', 'count_of_readm_measures_no_different', 'count_of_readm_measures_worse', 'readm_group_footnote', 'pt_exp_group_measure_count', 'count_of_facility_pt_exp_measures', 'pt_exp_group_footnote', 'te_group_measure_co

In [36]:
df_raw.head(10)

,facility_id,facility_name,address,citytown,state,zip_code,countyparish,telephone_number,hospital_type,hospital_ownership,...,count_of_readm_measures_better,count_of_readm_measures_no_different,count_of_readm_measures_worse,readm_group_footnote,pt_exp_group_measure_count,count_of_facility_pt_exp_measures,pt_exp_group_footnote,te_group_measure_count,count_of_facility_te_measures,te_group_footnote
0,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,Acute Care Hospitals,Government - Hospital District or Authority,...,1,9,1,,15,15,,10,10,
1,010005,MARSHALL MEDICAL CENTERS,2505 U S HIGHWAY 431 NORTH,BOAZ,AL,35957,MARSHALL,(256) 593-8310,Acute Care Hospitals,Government - Hospital District or Authority,...,1,8,0,,15,15,,10,10,
2,010006,NORTH ALABAMA MEDICAL CENTER,1701 VETERANS DRIVE,FLORENCE,AL,35630,LAUDERDALE,(256) 768-8400,Acute Care Hospitals,Proprietary,...,1,8,0,,15,15,,10,9,
3,010007,MIZELL MEMORIAL HOSPITAL,702 N MAIN ST,OPP,AL,36467,COVINGTON,(334) 493-3541,Acute Care Hospitals,Voluntary non-profit - Private,...,0,3,2,,15,5,,10,7,
4,010011,ST. VINCENT'S EAST,50 MEDICAL PARK EAST DRIVE,BIRMINGHAM,AL,35235,JEFFERSON,(205) 838-3122,Acute Care Hospitals,Voluntary non-profit - Private,...,1,5,2,29,15,10,29,10,7,29
5,010012,DEKALB REGIONAL MEDICAL CENTER,200 MED CENTER DRIVE,FORT PAYNE,AL,35968,DE KALB,(256) 845-3150,Acute Care Hospitals,Proprietary,...,0,6,2,,15,15,,10,9,
6,010016,SHELBY BAPTIST MEDICAL CENTER,1000 FIRST STREET NORTH,ALABASTER,AL,35007,SHELBY,(205) 620-8100,Acute Care Hospitals,Voluntary non-profit - Private,...,0,8,1,,15,15,,10,6,
7,010018,UAB CALLAHAN EYE HOSPITAL AUTHORITY,1720 UNIVERSITY BLVD STE 305,BIRMINGHAM,AL,35233,JEFFERSON,(205) 325-8596,Acute Care Hospitals,Voluntary non-profit - Private,...,0,1,0,,15,5,,10,3,
8,010019,HELEN KELLER HOSPITAL,1300 SOUTH MONTGOMERY AVENUE,SHEFFIELD,AL,35660,COLBERT,(256) 386-4556,Acute Care Hospitals,Government - Hospital District or Authority,...,1,5,1,,15,15,,10,9,
9,010021,DALE MEDICAL CENTER,126 HOSPITAL AVE,OZARK,AL,36360,DALE,(334) 774-2601,Acute Care Hospitals,Government - Hospital District or Authority,...,0,2,1,,15,5,,10,8,


In [37]:
missing = (
    df_raw.isna()
    .sum()
    .sort_values(ascending=False)
)
missing.head(15)

facility_id                                         0
facility_name                                       0
address                                             0
citytown                                            0
state                                               0
zip_code                                            0
countyparish                                        0
telephone_number                                    0
hospital_type                                       0
hospital_ownership                                  0
emergency_services                                  0
meets_criteria_for_birthing_friendly_designation    0
hospital_overall_rating                             0
hospital_overall_rating_footnote                    0
mort_group_measure_count                            0
dtype: int64

In [38]:
df_raw.duplicated().sum()

np.int64(0)

DATA INSIGHT

1. How many rows and columns were returned?

In [39]:
df_raw.shape

(4500, 38)

4500 rows and 38 columns were returned

2. Which columns appear important for the porject?

In [40]:
df_raw.columns.tolist()

['facility_id',
 'facility_name',
 'address',
 'citytown',
 'state',
 'zip_code',
 'countyparish',
 'telephone_number',
 'hospital_type',
 'hospital_ownership',
 'emergency_services',
 'meets_criteria_for_birthing_friendly_designation',
 'hospital_overall_rating',
 'hospital_overall_rating_footnote',
 'mort_group_measure_count',
 'count_of_facility_mort_measures',
 'count_of_mort_measures_better',
 'count_of_mort_measures_no_different',
 'count_of_mort_measures_worse',
 'mort_group_footnote',
 'safety_group_measure_count',
 'count_of_facility_safety_measures',
 'count_of_safety_measures_better',
 'count_of_safety_measures_no_different',
 'count_of_safety_measures_worse',
 'safety_group_footnote',
 'readm_group_measure_count',
 'count_of_facility_readm_measures',
 'count_of_readm_measures_better',
 'count_of_readm_measures_no_different',
 'count_of_readm_measures_worse',
 'readm_group_footnote',
 'pt_exp_group_measure_count',
 'count_of_facility_pt_exp_measures',
 'pt_exp_group_footnote

In [41]:
important_columns = [
    "facility_id",
    "facility_name",
    "citytown",
    "state",
    "hospital_type",
    "hospital_ownership",
    "hospital_overall_rating"
]

df_raw[important_columns].head()

,facility_id,facility_name,citytown,state,hospital_type,hospital_ownership,hospital_overall_rating
0,010001,SOUTHEAST HEALTH MEDICAL CENTER,DOTHAN,AL,Acute Care Hospitals,Government - Hospital District or Authority,4
1,010005,MARSHALL MEDICAL CENTERS,BOAZ,AL,Acute Care Hospitals,Government - Hospital District or Authority,3
2,010006,NORTH ALABAMA MEDICAL CENTER,FLORENCE,AL,Acute Care Hospitals,Proprietary,2
3,010007,MIZELL MEMORIAL HOSPITAL,OPP,AL,Acute Care Hospitals,Voluntary non-profit - Private,1
4,010011,ST. VINCENT'S EAST,BIRMINGHAM,AL,Acute Care Hospitals,Voluntary non-profit - Private,3


Important columns include facility ID, facility name, state, hospital type, hospital ownership, and hospital overall rating because these fields allow hospitals to be identified and compared actoss the United States. 

3. How is the hospital overall rating stored?

In [42]:
df_raw["hospital_overall_rating"].dtype

<StringDtype(storage='python', na_value=nan)>

In [43]:
df_raw["hospital_overall_rating"].value_counts(dropna=False)

hospital_overall_rating
Not Available    1819
3                 837
4                 780
2                 589
5                 294
1                 181
Name: count, dtype: int64

Hospital overall rating is stored as a text/object variable because the column contains both numerical ratings and values such as "Not Available". 

4. Are missing values represented as blanks, text, or another value?

In [44]:
df_raw.isna().sum().sort_values(ascending=False).head(10)

facility_id           0
facility_name         0
address               0
citytown              0
state                 0
zip_code              0
countyparish          0
telephone_number      0
hospital_type         0
hospital_ownership    0
dtype: int64

In [45]:
(df_raw == "Not Available").sum().sort_values(ascending=False).head(10)

hospital_overall_rating                  1819
count_of_facility_safety_measures        1704
count_of_safety_measures_better          1704
count_of_safety_measures_no_different    1704
count_of_safety_measures_worse           1704
count_of_facility_pt_exp_measures        1609
count_of_mort_measures_better            1061
count_of_mort_measures_worse             1061
count_of_facility_mort_measures          1061
count_of_mort_measures_no_different      1061
dtype: int64

Missing or unavailable information is represented in multiple ways, including text such as "Not Available" and potentially blank or null values. 

5. Are there duplicate rows?

In [46]:
df_raw.duplicated().sum()

np.int64(0)

In [47]:
df_raw["facility_id"].duplicated().sum()

np.int64(0)

No fully duplicated rows or duplicate facility IDs were identified. 

6. Which categories exist for hospital type and ownership?

Hospital Type

In [48]:
df_raw["hospital_type"].value_counts(dropna=False)

hospital_type
Acute Care Hospitals                    2602
Critical Access Hospitals               1119
Psychiatric                              531
Acute Care - Veterans Administration     111
Childrens                                 70
Rural Emergency Hospital                  37
Acute Care - Department of Defense        26
Long-term                                  4
Name: count, dtype: int64

Hospital Ownership 

In [49]:
df_raw["hospital_ownership"].value_counts(dropna=False)

hospital_ownership
Voluntary non-profit - Private                 1953
Proprietary                                     842
Government - Hospital District or Authority     380
Government - Local                              362
Voluntary non-profit - Other                    304
Voluntary non-profit - Church                   235
Government - State                              177
Veterans Health Administration                  111
Physician                                        51
Government - Federal                             43
Department of Defense                            26
Tribal                                           16
Name: count, dtype: int64

## Data Cleaning and Validation

The CMS hospital dataset is cleaned and validated before analysis. This includes reviewing missing values, converting hospital ratings to a numeric format, checking duplicate records, and confirming that key variables are suitable for analysis.

1. Making a working copy

In [50]:
df=df_raw.copy()

2. Normalizing column names

In [51]:
df.columns=(
    df.columns
     .str.strip()
     .str.lower()
     .str.replace(r"[^a-z0-9]+","_",regex=True)
     .str.strip("_")
)

df.columns.tolist()


['facility_id',
 'facility_name',
 'address',
 'citytown',
 'state',
 'zip_code',
 'countyparish',
 'telephone_number',
 'hospital_type',
 'hospital_ownership',
 'emergency_services',
 'meets_criteria_for_birthing_friendly_designation',
 'hospital_overall_rating',
 'hospital_overall_rating_footnote',
 'mort_group_measure_count',
 'count_of_facility_mort_measures',
 'count_of_mort_measures_better',
 'count_of_mort_measures_no_different',
 'count_of_mort_measures_worse',
 'mort_group_footnote',
 'safety_group_measure_count',
 'count_of_facility_safety_measures',
 'count_of_safety_measures_better',
 'count_of_safety_measures_no_different',
 'count_of_safety_measures_worse',
 'safety_group_footnote',
 'readm_group_measure_count',
 'count_of_facility_readm_measures',
 'count_of_readm_measures_better',
 'count_of_readm_measures_no_different',
 'count_of_readm_measures_worse',
 'readm_group_footnote',
 'pt_exp_group_measure_count',
 'count_of_facility_pt_exp_measures',
 'pt_exp_group_footnote

3. Preserve Facility ID as text

In [52]:
df["facility_id"]=df["facility_id"].astype("string")

4. Concert the overall rating to numeric

In [53]:
df["hospital_overall_rating"]=pd.to_numeric(
    df["hospital_overall_rating"],
    errors="coerce"
)

errors = "coerce" turns nonnumeric values such as unavailable text into NaN (missing). Better than using 0 since the rating scale is from 1-5. 

5. Standardize state text

In [54]:
df["state"]=(
    df["state"]
     .astype("string")
     .str.strip()
     .str.upper()
)

6. Validating rating range

In [55]:
valid_rating = df["hospital_overall_rating"].dropna().between(1, 5)
assert valid_rating.all(), "A rating outside the expected 1-5 range was found."
print(df["hospital_overall_rating"].value_counts(dropna=False).sort_index())

hospital_overall_rating
1.0     181
2.0     589
3.0     837
4.0     780
5.0     294
NaN    1819
Name: count, dtype: int64


7. Checking duplicate facility identifiers

In [56]:
duplicate_facilities = df["facility_id"].duplicated(keep=False)
print("Duplicate Facility IDs:", duplicate_facilities.sum())
df.loc[duplicate_facilities, [
    "facility_id", "facility_name", "state"
]].head(20)

Duplicate Facility IDs: 0


,facility_id,facility_name,state


No duplicates found for facility identifiers(facility_id)

## Analysis Table creation 

In [57]:
core_columns = [
    "facility_id",
    "facility_name",
    "city_town",
    "state",
    "hospital_type",
    "hospital_ownership",
    "emergency_services",
    "hospital_overall_rating",
]
available_core = [c for c in core_columns if c in df.columns]
hospitals = df[available_core].copy()

print(hospitals.shape)
hospitals.head()

(4500, 7)


,facility_id,facility_name,state,hospital_type,hospital_ownership,emergency_services,hospital_overall_rating
0,010001,SOUTHEAST HEALTH MEDICAL CENTER,AL,Acute Care Hospitals,Government - Hospital District or Authority,Yes,4.0
1,010005,MARSHALL MEDICAL CENTERS,AL,Acute Care Hospitals,Government - Hospital District or Authority,Yes,3.0
2,010006,NORTH ALABAMA MEDICAL CENTER,AL,Acute Care Hospitals,Proprietary,Yes,2.0
3,010007,MIZELL MEMORIAL HOSPITAL,AL,Acute Care Hospitals,Voluntary non-profit - Private,Yes,1.0
4,010011,ST. VINCENT'S EAST,AL,Acute Care Hospitals,Voluntary non-profit - Private,Yes,3.0


1. Adding data-quality summary values

In [58]:
quality_summary = {
    "rows": len(hospitals),
    "unique_facilities": hospitals["facility_id"].nunique(),
    "rated_hospitals": hospitals["hospital_overall_rating"].notna().sum(),
    "missing_ratings": hospitals["hospital_overall_rating"].isna().sum(),
}
quality_summary

{'rows': 4500,
 'unique_facilities': 4500,
 'rated_hospitals': np.int64(2681),
 'missing_ratings': np.int64(1819)}

2. Creating processed CSV 

This creates a file inside GitHub Codespace. 

In [59]:
from pathlib import Path
Path("data/processed").mkdir(parents=True, exist_ok=True)
hospitals.to_csv("data/processed/hospitals_clean.csv", index=False)